# SPM 3405: Machine Learning — Lecture 4 notebook
## Resampling and regularisation

**Dr Charles Ndung'u** · Department of Physics, University of Nairobi

In this notebook you will:
1. Choose the degree of a thermistor calibration with the **validation set**, **LOOCV** and **10-fold cross-validation**.
2. Use the **bootstrap** to find the uncertainty of the 50% vibration point from Lecture 3.
3. Fit an electronic nose with 40 channels using **least squares, ridge and the lasso**, choosing $\lambda$ by cross-validation.

Reading: ISLP 5.1–5.2 and 6.1–6.2. Run each cell with **Shift + Enter**, from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score, train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(405)   # fixed seed: everyone gets the same numbers as the slides

## Part 1: Calibrating a thermistor

A 10 kΩ NTC thermistor: $R = 10\,\exp[B(1/T_K - 1/298.15)]$ kΩ with $B = 3950$ K and $T_K$ in kelvin.
We record $R$ (1% measurement noise) against a reference thermometer (0.5 °C noise) at 40 temperatures, and predict $T$ from $x = \ln R$
with a polynomial of degree $d$. Which $d$?

In [ ]:
n = 40
T = rng.uniform(0, 100, n)
R = 10.0 * np.exp(3950 * (1 / (T + 273.15) - 1 / 298.15))      # kOhm
x = np.log(R * (1 + rng.normal(0, 0.01, n)))                    # measured ln R
y = T + rng.normal(0, 0.5, n)                                   # reference temperature (deg C)

plt.semilogx(R, y, "o"); plt.xlabel("R (kΩ)"); plt.ylabel("T (°C)"); plt.show()

def poly(d):
    return make_pipeline(PolynomialFeatures(d), LinearRegression())
degrees = range(1, 11)

### The validation set approach: 10 random halves

In [ ]:
val = []
for rep in range(10):
    tr, va = train_test_split(np.arange(n), test_size=0.5, random_state=rep)
    val.append([np.mean((poly(d).fit(x[tr, None], y[tr]).predict(x[va, None]) - y[va])**2) for d in degrees])
val = np.array(val)
print("best degree for each split:", val.argmin(axis=1) + 1)
for curve in val:
    plt.semilogy(degrees, curve)
plt.xlabel("degree"); plt.ylabel("validation MSE"); plt.show()

### LOOCV and 10-fold cross-validation, compared with the true test error

In [ ]:
loo = [-cross_val_score(poly(d), x[:, None], y, cv=LeaveOneOut(), scoring="neg_mean_squared_error").mean() for d in degrees]
k10 = [-cross_val_score(poly(d), x[:, None], y, cv=KFold(10, shuffle=True, random_state=0),
                        scoring="neg_mean_squared_error").mean() for d in degrees]

# 5000 new readings, only possible because the data are simulated
Tt = rng.uniform(0, 100, 5000)
Rt = 10.0 * np.exp(3950 * (1 / (Tt + 273.15) - 1 / 298.15))
xt = np.log(Rt * (1 + rng.normal(0, 0.01, 5000)))
yt = Tt + rng.normal(0, 0.5, 5000)
test = [np.mean((poly(d).fit(x[:, None], y).predict(xt[:, None]) - yt)**2) for d in degrees]
train = [np.mean((poly(d).fit(x[:, None], y).predict(x[:, None]) - y)**2) for d in degrees]

for d, a, b, c, e in zip(degrees, train, loo, k10, test):
    print(f"degree {d:2d}: train {a:6.3f}  LOOCV {b:6.3f}  10-fold {c:6.3f}  true test {e:6.3f}")
print("LOOCV picks", np.argmin(loo) + 1, "; 10-fold picks", np.argmin(k10) + 1, "; true best", np.argmin(test) + 1)

plt.semilogy(degrees, train, "o-", label="training")
plt.semilogy(degrees, loo, "s-", label="LOOCV")
plt.semilogy(degrees, k10, "^-", label="10-fold CV")
plt.semilogy(degrees, test, "k--", label="true test")
plt.xlabel("degree"); plt.ylabel("MSE (°C²)"); plt.legend(); plt.show()

**Questions**
1. Why do the 10 validation-set curves disagree so much about the best degree?
2. Why does the training MSE keep falling while the CV error rises after degree 3?
3. Physics says $1/T_K$ is almost linear in $\ln R$ (the Steinhart–Hart equation). Does a low degree make sense?

## Part 2: The bootstrap — how sure is the 50% vibration point?

We regenerate the 200 motors of Lecture 3 exactly (same seed) and ask: how uncertain is $v_{50} = -\hat\beta_0/\hat\beta_1$?

In [ ]:
r3 = np.random.default_rng(405)                     # the Lecture 3 generator
sig = lambda z: 1 / (1 + np.exp(-z))
vib = r3.uniform(0, 10, 200)
fault = (r3.uniform(size=200) < sig(-6 + 1.5 * vib)).astype(int)

def v50(v, f):
    m = LogisticRegression(C=np.inf, max_iter=1000).fit(v[:, None], f)
    return -m.intercept_[0] / m.coef_[0, 0]

estimate = v50(vib, fault)
boot = []
for b in range(1000):
    i = rng.integers(0, 200, 200)                   # 200 rows drawn WITH replacement
    boot.append(v50(vib[i], fault[i]))
boot = np.array(boot)
print(f"estimate {estimate:.2f} mm/s, bootstrap SE {boot.std(ddof=1):.2f} mm/s")
print("95% interval (2.5% and 97.5% percentiles):", np.percentile(boot, [2.5, 97.5]).round(2))
plt.hist(boot, bins=20); plt.axvline(4.0, color="k", ls="--", label="true value")
plt.xlabel("50% point (mm/s)"); plt.legend(); plt.show()

**Question 4.** In one bootstrap sample of 200 rows, about how many of the original motors are missing? (Hint: $(1 - 1/n)^n \approx e^{-1}$.) Check it with `len(np.unique(i))`.

## Part 3: An electronic nose — ridge and lasso

40 gas-sensor channels, 50 training samples, 1000 test samples. Humidity drift affects every channel a little.
Only channels 3, 11, 19, 27 and 35 are related to the adulterant concentration.

In [ ]:
p = 40
def enose(m):
    common = rng.normal(0, 1, (m, 1))               # humidity drift, shared by every channel
    return 0.7 * common + rng.normal(0, 1, (m, p))
X_train = enose(50); X_test = enose(1000)
beta = np.zeros(p); beta[[3, 11, 19, 27, 35]] = [1.5, -1.0, 0.8, 1.2, -0.7]
y_train = 10 + X_train @ beta + rng.normal(0, 1, 50)
y_test = 10 + X_test @ beta + rng.normal(0, 1, 1000)

scaler = StandardScaler().fit(X_train)              # standardise with the TRAINING data only
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)
mse = lambda a, b: np.mean((a - b)**2)

ls = LinearRegression().fit(Z_train, y_train)
print(f"guess the average: test {mse(y_train.mean(), y_test):.2f}")
print(f"least squares:     train {mse(ls.predict(Z_train), y_train):.2f}  test {mse(ls.predict(Z_test), y_test):.2f}")

ridge = RidgeCV(alphas=np.logspace(-3, 3, 60), cv=10).fit(Z_train, y_train)
print(f"ridge (CV):        lambda {ridge.alpha_:.1f}  test {mse(ridge.predict(Z_test), y_test):.2f}")

lasso = LassoCV(alphas=np.logspace(-3, 1, 60), cv=KFold(10, shuffle=True, random_state=0), max_iter=100000).fit(Z_train, y_train)
kept = np.flatnonzero(lasso.coef_)
print(f"lasso (CV):        lambda {lasso.alpha_:.3f}  test {mse(lasso.predict(Z_test), y_test):.2f}  channels kept: {len(kept)}")
print("kept channels:", kept.tolist())
print("lasso coefficients of the true channels:", lasso.coef_[[3, 11, 19, 27, 35]].round(2))
others = np.setdiff1d(kept, [3, 11, 19, 27, 35])
print("largest other coefficient:", np.abs(lasso.coef_[others]).max().round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
lams = np.logspace(-2, 4, 40)
ax[0].semilogx(lams, [Ridge(alpha=a).fit(Z_train, y_train).coef_ for a in lams]); ax[0].set_title("ridge path")
lams = np.logspace(-2, 1, 40)
ax[1].semilogx(lams, [Lasso(alpha=a, max_iter=100000).fit(Z_train, y_train).coef_ for a in lams]); ax[1].set_title("lasso path")
for a in ax: a.set_xlabel("lambda"); a.set_ylabel("coefficient")
plt.tight_layout(); plt.show()

**Questions**
5. Least squares has training MSE 0.22 but test MSE 4.61. Explain with the bias–variance trade-off.
6. Which method would you report to a dairy engineer, and why? Which channels would you tell them to keep?
7. Remove the line that standardises the channels and refit the lasso. Does anything change here? When would it matter?
8. **Your group project:** estimate your baseline model's error by 10-fold CV (`cross_val_score`). If you have many inputs, try `LassoCV`.

## Your turn: group tasks

Work on these in your group. Write the code in the empty cells and answer each question in a text cell below it,
in your own words. Submit **one notebook per group** on Google Classroom, with every cell run.
Every member must be able to explain every line.

The tasks reuse `x`, `y`, `poly`, `degrees`, `mse` and the imports from above, so run the whole notebook first.

### Task 1: Write cross-validation yourself

`cross_val_score` hides the loop. Write it out, so that you know exactly what it does.

(a) Write a function `kfold_mse(d, k)` that splits the 40 thermistor readings into `k` folds (shuffle the indices first with
`np.random.default_rng(0).permutation(n)` and cut them with `np.array_split`). For each fold, fit `poly(d)` on the other folds,
predict the held-out fold, and record its MSE. Return the average over the folds.

(b) Use it to compute the 5-fold and 10-fold CV error for degrees 1 to 10. Which degree does each choose?
Compare with the 10-fold column printed in Part 1 (it will not match exactly: the folds differ). The CV error is almost flat over
several degrees: which degree would you report to an instrument maker, and why?

(c) How many times is the model fitted for 10-fold CV over 10 degrees? And for LOOCV? Why does that matter for big models?

In [ ]:
# Task 1: your code here

### Task 2: The bootstrap by hand, for a sensor

A pressure sensor is calibrated at 15 reference pressures. The cell below makes the data; do not change it.

(a) Fit the straight line $V = \beta_0 + \beta_1 p$ and find the pressure that gives 2.5 V: $p^* = (2.5 - \hat\beta_0)/\hat\beta_1$.

(b) Bootstrap: 2000 times, draw 15 rows **with replacement**, refit the line and recompute $\hat\beta_1$ and $p^*$.
Give the bootstrap standard error of each, and a 95% interval for $p^*$ (2.5% and 97.5% percentiles).

(c) For the slope there is also a formula: $\mathrm{SE}(\hat\beta_1) = \hat\sigma / \sqrt{\sum_i (p_i - \bar p)^2}$, with
$\hat\sigma^2 = \mathrm{RSS}/(n-2)$. Compute it and compare with the bootstrap. Why is the bootstrap the only easy option for $p^*$?

In [ ]:
rng4 = np.random.default_rng(4055)                # task data: do not change
pr = np.sort(rng4.uniform(0, 200, 15))            # reference pressure (kPa)
Vp = 0.5 + 0.02 * pr + rng4.normal(0, 0.05, 15)   # sensor output (V)

In [ ]:
# Task 2: your code here

### Task 3: Which wavelengths measure fat in milk?

A near-infrared spectrometer measures the absorbance of a milk sample at 100 wavelengths. We want to predict the fat content (%)
from the spectrum, with only 60 calibrated samples: more inputs than samples ($p = 100 > n = 60$).
Neighbouring wavelengths are strongly correlated. The cell below makes the data; do not change it.

(a) Standardise the spectra (fit the scaler on the training samples only). Fit least squares, `RidgeCV` and `LassoCV` and give the
test MSE of each. What goes wrong with least squares when $p > n$?

(b) Plot the lasso coefficients against wavelength. Which wavelengths does it keep? Compare with `true_bands` (hidden in a real
experiment). Why does it also keep some wavelengths far from the fat bands? (Hint: look at `base` in `milk_spectra`.)

(c) Ridge or lasso: which would you give to the dairy, and why?

In [ ]:
rng5 = np.random.default_rng(4056)                # task data: do not change
wl = np.linspace(900, 1700, 100)                   # wavelength (nm)
true_bands = [1210, 1450]                          # where fat absorbs (nm)
def milk_spectra(m):
    fat = rng5.uniform(2, 6, m)                    # fat content (%)
    base = rng5.normal(0, 1, (m, 1)) * np.exp(-((wl - 1300) / 400)**2)          # scattering baseline
    peaks = sum(np.exp(-((wl - c) / 25)**2) for c in true_bands)
    S = 0.3 * fat[:, None] * peaks + base + rng5.normal(0, 0.2, (m, 100))
    return S, fat
S_train, fat_train = milk_spectra(60)
S_test, fat_test = milk_spectra(1000)

In [ ]:
# Task 3: your code here